In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re 
import glob
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
import matplotlib.ticker as ticker

In [2]:
# Read in data
df = pd.read_csv('../data/clustered_dr_all_cells_2022_09_15.csv', index_col=0)
df.head()

,209Bi_CD45,Center,164Dy,166Er_CD34,Event_length,157Gd,113In_CD45,191Ir_DNA1,193Ir_DNA2,104Pd_CD45,...,171Yb_Granzyme_B_asinh_coarseAlign_fineAlign,172Yb_CD38_asinh_coarseAlign_fineAlign,173Yb_CD14_asinh_coarseAlign_fineAlign,174Yb_HLA-DR_asinh_coarseAlign_fineAlign,176Yb_CD56_asinh_coarseAlign_fineAlign,Alignment_MC_fineAlign,FlowSOM_cluster,FlowSOM_metacluster,UMAP_X,UMAP_Y
1,0.00000,633.973,0.0,0.00000,15,0.0,0.0,249.076,400.114,0.0,...,2.951135,0.961040,0.047869,0.053259,0.044064,5,179,39,0.075119,9.367344
2,0.00000,855.374,0.0,0.00000,17,0.0,0.0,218.016,455.480,0.0,...,0.113675,1.902335,1.078308,2.566125,0.085914,1,35,2,-0.657520,-6.955027
3,2.47164,625.283,0.0,0.00000,15,0.0,0.0,223.581,583.897,0.0,...,5.450052,1.786209,0.009813,0.905944,3.293293,4,186,33,-10.401136,3.650752
4,0.00000,795.305,0.0,0.00000,17,0.0,0.0,189.046,480.253,0.0,...,5.067924,1.552994,0.014530,0.073580,1.197077,7,192,39,-0.358508,9.502626
5,0.00000,0.000,0.0,1.32702,16,0.0,0.0,118.551,241.897,0.0,...,0.709685,3.955264,0.255739,0.540499,0.085914,1,9,3,0.421227,-3.556041


In [3]:
# Read in patient alias
patient_alias = pd.read_excel('../data/patient_alias_from_2025.xlsx', nrows= 40)
patient_alias_dict = dict(zip(patient_alias['PID'].astype(str), patient_alias['alias']))
patient_alias_dict

control_alias = pd.read_csv('../data/healthy_control.csv')
control_alias['key'] = control_alias['key'].str.replace('Healthy_BMA_67y_Male', 'Healthy_BMA_67Y_Male')

control_alias_dict = dict(zip(control_alias['key'].str.replace('_T_Cell_Panel.fcs', ''), control_alias['value'].str.replace('.fcs', '')))

In [4]:
# to fix 
df['FileName'], df['Patient_ID']

(1             61201_001_C1_D8_T_Cell_Panel
 2             61201_001_C1_D8_T_Cell_Panel
 3             61201_001_C1_D8_T_Cell_Panel
 4             61201_001_C1_D8_T_Cell_Panel
 5             61201_001_C1_D8_T_Cell_Panel
                         ...               
 294996    Healthy_BMA_HSA4848_T_Cell_Panel
 294997    Healthy_BMA_HSA4848_T_Cell_Panel
 294998    Healthy_BMA_HSA4848_T_Cell_Panel
 294999    Healthy_BMA_HSA4848_T_Cell_Panel
 295000    Healthy_BMA_HSA4848_T_Cell_Panel
 Name: FileName, Length: 295000, dtype: object,
 1                   61201_001
 2                   61201_001
 3                   61201_001
 4                   61201_001
 5                   61201_001
                  ...         
 294996    Healthy_BMA_HSA4848
 294997    Healthy_BMA_HSA4848
 294998    Healthy_BMA_HSA4848
 294999    Healthy_BMA_HSA4848
 295000    Healthy_BMA_HSA4848
 Name: Patient_ID, Length: 295000, dtype: object)

In [5]:
# Rename the filenames
filename_dict = {}
for i in df['FileName'].unique():
    if i.startswith('6'):
        x = i.split('_')[0] + '_' + i.split('_')[1]
        x = x.replace('_', '')
        z = '_'.join(i.split('_')[2:])
        filename_dict[i] = patient_alias_dict[x] + '_' + z
    elif i.startswith('Healthy'):
        #x = i.split('_')[0] + '_' + i.split('_')[1] + '_' + i.split('_')[2]
        x = i.replace('_T_Cell_Panel', '')
        filename_dict[i] = control_alias_dict[x] + '_T_Cell_Panel'

df['FileName'] = df['FileName'].map(filename_dict)

In [6]:
# Rename the filename column


In [7]:
# Rename the filenames
patient_id_dict = {}
for i in df['Patient_ID'].unique():
    if i.startswith('6'):
        patient_id_dict[i] = patient_alias_dict[i.replace('_', '').replace('612500042', '61250004')]
    elif i.startswith('Healthy'):
        #x = i.split('_')[0] + '_' + i.split('_')[1] + '_' + i.split('_')[2]
        x = i.replace('_T_Cell_Panel', '')
        patient_id_dict[i] = control_alias_dict[x]

df['Patient_ID'] = df['Patient_ID'].map(patient_id_dict)

In [8]:
df['Patient_ID'].unique()

array(['P08', 'P24', 'P19', 'P12', 'P04', 'P03', 'P27', 'P38', 'P06',
       'P10', 'P16', 'P14', 'P05', 'P02', 'P01', 'P11', 'P28', 'P07',
       'P15', 'P20', 'P26', 'P17', 'P18', 'P22', 'P25', 'P13', 'P21',
       'Control_14', 'Control_1', 'Control_17', 'Control_19',
       'Control_13', 'Control_18', 'Control_9', 'Control_7', 'Control_15',
       'Control_3', 'Control_2', 'Control_6', 'Control_8', 'Control_5',
       'Control_16', 'Control_10', 'Control_4', 'Control_12',
       'Control_11'], dtype=object)

In [9]:
df['FileName'].unique()

array(['P08_C1_D8_T_Cell_Panel', 'P08_C7_D1_T_Cell_Panel',
       'P08_C7_D22_T_Cell_Panel', 'P08_SPD_T_Cell_Panel',
       'P24_C1_D1_T_Cell_Panel', 'P24_C1_D8_T_Cell_Panel',
       'P24_C7_D1_T_Cell_Panel', 'P19_C1_D1_T_Cell_Panel',
       'P19_C1_D8_T_Cell_Panel', 'P19_C7_D1_T_Cell_Panel',
       'P19_C7_D22_T_Cell_Panel', 'P12_C1_D1_T_Cell_Panel',
       'P12_C1_D8_T_Cell_Panel', 'P12_C12_D29_T_Cell_Panel',
       'P12_C7_D22_T_Cell_Panel', 'P04_C1_D1_T_Cell_Panel',
       'P04_C1_D8_T_Cell_Panel', 'P04_C12_D29_T_Cell_Panel',
       'P04_C7_D22_T_Cell_Panel', 'P03_C1_D1_T_Cell_Panel',
       'P03_C1_D8_T_Cell_Panel', 'P03_C12_D29_T_Cell_Panel',
       'P03_C7_D1_T_Cell_Panel', 'P03_C7_D22_T_Cell_Panel',
       'P27_C1_D1_T_Cell_Panel', 'P38_C1_D1_T_Cell_Panel',
       'P38_C1_D8_T_Cell_Panel', 'P06_C12_D29_T_Cell_Panel',
       'P06_C7_D1_T_Cell_Panel', 'P06_C7_D22_T_Cell_Panel',
       'P10_C1_D1_T_Cell_Panel', 'P10_C1_D8_T_Cell_Panel',
       'P10_C12_D29_T_Cell_Panel', 'P10_C7_D

In [22]:
df.to_csv('../data/clustered_dr_all_cells_2022_09_15_patient_updated.csv')